# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates loading, exploration, and basic analysis of the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. The workflow follows best practices for referencing Croissant entities (record sets, fields, and columns) **by their `@id` fields** for complete reproducibility.

### Dataset Source
The dataset source is provided via a Croissant schema JSON-LD URL:

In [ ]:
# Ensure mlcroissant is installed
!pip install -U mlcroissant

## 1. Data Loading

First, load the Croissant schema and metadata for the FAIR² dataset.

**Note**: All accesses to entities use their `@id` for clarity and reproducibility.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load Croissant dataset metadata and record structure
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")


## 2. Data Overview

Explore the available record sets, fields, and their corresponding `@id`s as defined in the schema. All navigation and identification uses `@id` strings.

First, enumerate all record sets (tables/collections) in the dataset.

In [ ]:
# List all record sets and their fields by @id
def list_record_sets_and_fields(ds):
    record_sets = ds.metadata.record_sets
    print(f"\nFound {len(record_sets)} record sets:")
    for rs in record_sets:
        print(f"- Record set: {{rs['@id']}} (name: {rs.get('name', '')})")
        # fields is a list of dicts, each with @id
        fields = rs.get('fields', [])
        if fields:
            print(f"    Fields:")
            for f in fields:
                print(f"      - {{f['@id']}} (name: {f.get('name', '')}, dataType: {f.get('dataType', '')})")

list_record_sets_and_fields(dataset)


Alternatively, let's collect all record set `@id`s for use in extraction. If you wish to see individual examples from each set, the following code prints the first record of each.

In [ ]:
# Preview the first record from each record set, using `@id` reference
for rs in dataset.metadata.record_sets:
    rs_id = rs["@id"]
    print(f"\nRecord set @id: {rs_id}")
    try:
        records = list(dataset.records(record_set=rs_id))
        if records:
            print(f"First record:\n{records[0]}")
        else:
            print("[Empty set]")
    except Exception as e:
        print(f"Failed to load records: {e}")

## 3. Data Extraction

Choose relevant record sets (by `@id`) and load them as pandas DataFrames for further processing. All variables correspond to schema entity `@id`s.

**Note**: Since there may be several record sets, we load all of them and store as a dictionary for referencing by `@id`.

In [ ]:
# Get all record set @ids
record_set_ids = [rs["@id"] for rs in dataset.metadata.record_sets]
dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
    else:
        df = pd.DataFrame()
    dataframes[record_set_id] = df

# Print the columns of the main (largest) record set
main_rs_id = max(dataframes, key=lambda x: dataframes[x].shape[0]) if dataframes else None
if main_rs_id:
    print(f"Main record set: {main_rs_id}")
    print("Columns:", dataframes[main_rs_id].columns.tolist())
    display(dataframes[main_rs_id].head())
else:
    print("No non-empty record sets found.")

## 4. Exploratory Data Analysis (EDA)

*Reference everything by `@id`!*

We'll select a numeric field to analyze (e.g., `age_at_second_crc_diagnosis`). The following block demonstrates filtering, normalization, and grouping by another attribute. Ensure you use actual field `@id`s and not column names; if needed, map to field names using the Croissant metadata above.

In [ ]:
# For demonstration, identify a numeric field from the metadata of the main record set
main_fields = []
for rs in dataset.metadata.record_sets:
    if rs["@id"] == main_rs_id:
        main_fields = rs.get("fields", [])
        break
numeric_field_id = None
for f in main_fields:
    if f.get("dataType") in ["schema:Float", "schema:Integer", "schema:Number"]:
        numeric_field_id = f["@id"]
        print(f"Using numeric field: {numeric_field_id} ({f.get('name', '')})")
        break

# Pick a group field (categorical) for demonstration
group_field_id = None
for f in main_fields:
    if f.get("dataType") == "schema:Text":
        group_field_id = f["@id"]
        print(f"Grouping by: {group_field_id} ({f.get('name', '')})")
        break

df = dataframes[main_rs_id]

if numeric_field_id and numeric_field_id in df.columns:
    # Remove nulls
    df_num = df[df[numeric_field_id].notnull()].copy()
    threshold = df_num[numeric_field_id].quantile(0.10)
    filtered_df = df_num[df_num[numeric_field_id] > threshold].copy()
    print(f"Filtered {numeric_field_id} records above the 10th percentile ({threshold:.2f}): {len(filtered_df)} rows")
    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
        filtered_df[numeric_field_id].std()
    )
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Optional: grouping
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped mean {numeric_field_id} by {group_field_id}:")
        display(grouped_df.head())
else:
    print("No numeric field found for EDA with data in main DataFrame.")

## 5. Visualization

Visualize data distributions or relationships.

- Histogram of the selected numeric field
- Boxplot by group (if grouping variable is available, as above)

<sub>**Visualization always references columns by their `@id`**</sub>

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=12, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

if numeric_field_id and group_field_id and group_field_id in df.columns:
    plt.figure(figsize=(10, 5))
    sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 6. Conclusion

* This notebook demonstrates loading the FAIR² dataset via Croissant and exploring its schema and records **using `@id` references throughout**.
* We performed basic filtering and normalization on a numeric record set field, and visualized the distributions using `matplotlib` and `seaborn`.
* The workflow can be extended by referencing further entities via their Croissant `@id` fields for robust, reusable analysis workflows.

**Remember**: For all production data work, always use entity `@id`s for field, record set, and column selection and referencing to ensure reproducibility and future-compatibility.
